# EIS Scenario Forecast — Config-Driven

This notebook runs any EIS housing scenario by reading all parameters from a JSON config file.

**The only cell you need to edit is the one below the "Load Scenario Configuration" heading** — set `config_path` to point at your scenario's config file, then run all cells.

Config files live in `Forecast/configs/`. Each contains all scenario parameters including the residential unit counts — no external CSVs needed.

---

### Config file format

```json
{
    "scenario_name": "Alternative_5_reduced_",
    "adjust_for_workfromhome": "no",
    "adjust_occupancy": "yes",
    "adjust_occupancy_additional_units": 855,
    "adjust_taz_population": "no",
    "zone_proportions": {
        "default": {"mf": 0.35, "sf": 0.50, "infill": 0.15}
    },
    "occupancy_rates": {
        "Bonus": 1.00, "CTC": 1.00, "General": 0.35,
        "ADU": 0.70, "Affordable": 1.00, "Moderate": 1.00
    },
    "income_proportions": {
        "Bonus":      {"low": 0.40, "medium": 0.25, "high": 0.35},
        "General":    {"low": 0.29, "medium": 0.18, "high": 0.53},
        "ADU":        {"low": 0.65, "medium": 0.20, "high": 0.15},
        "CTC":        {"low": 1.00, "medium": 0.00, "high": 0.00},
        "Affordable": {"low": 0.40, "medium": 0.25, "high": 0.35},
        "Moderate":   {"low": 0.29, "medium": 0.18, "high": 0.53}
    },
    "residential_units": [
        {"Jurisdiction": "TRPA", "Unit_Pool": "ADU",      "Future_Units": 110},
        {"Jurisdiction": "CSLT", "Unit_Pool": "Bonus Unit","Future_Units": 89},
        {"Jurisdiction": "TRPA", "Unit_Pool": "General",  "Future_Units": 948}
    ]
}
```

### Optional fields

| Field | Default | Description |
|---|---|---|
| `adjust_occupancy_additional_units` | — | Required when `adjust_occupancy` is `"yes"` |
| `adjust_taz_population` | `"no"` | If `"yes"`, calls `adjust_taz_population()` to hit regional targets instead of applying the existing persons-per-occ-unit lookup. Saves updated lookup CSVs to `base_data/` for downstream scenarios. |
| `target_population_2035` | — | Required when `adjust_taz_population` is `"yes"` |
| `target_population_2050` | — | Required when `adjust_taz_population` is `"yes"` |

# Establish Variables and Environment

In [ ]:
# import packages
import pandas as pd
import pathlib
from pathlib import Path
import os
import sys
import arcpy
import numpy as np
import pickle
import json
import shutil
# external connection packages
from sqlalchemy.engine import URL
from sqlalchemy import create_engine
# add scripts folder to path so utils and forecast_functions can be imported
sys.path.insert(0, str(pathlib.Path().absolute()))
from utils import *
from forecast_functions import *
# pandas options
pd.options.mode.copy_on_write = True
pd.options.mode.chained_assignment = None
pd.options.display.max_columns = 999
pd.options.display.max_rows    = 999

# my workspace
workspace = r"C:\GIS\Scratch.gdb"
# current working directory
local_path = pathlib.Path().absolute()

# set data path as a subfolder of the parent directory
data_dir      = local_path.parents[0] / 'data'
# folder to save processed data
out_dir       = local_path.parents[0] / 'data/processed_data'
base_data_dir = local_path.parents[0] / 'data/processed_data/base_data'

gdb = workspace
arcpy.env.workspace = 'memory'
arcpy.env.overwriteOutput = True
# Set spatial reference to NAD 1983 UTM Zone 10N
sr = arcpy.SpatialReference(26910)

# network path to connection files
filePath   = "F:/GIS/PARCELUPDATE/Workspace/"
sdeBase    = os.path.join(filePath, "Vector.sde")
sdeCollect = os.path.join(filePath, "Collection.sde")
sdeTabular = os.path.join(filePath, "Tabular.sde")
sdeEdit    = os.path.join(filePath, "Edit.sde")

initial_columns = [
    'APN',
    'APO_ADDRESS',
    'Residential_Units',
    'TouristAccommodation_Units',
    'CommercialFloorArea_SqFt',
    'YEAR',
    'JURISDICTION',
    'COUNTY',
    'OWNERSHIP_TYPE',
    'COUNTY_LANDUSE_DESCRIPTION',
    'EXISTING_LANDUSE',
    'REGIONAL_LANDUSE',
    'YEAR_BUILT',
    'PLAN_ID',
    'PLAN_NAME',
    'ZONING_ID',
    'ZONING_DESCRIPTION',
    'TOWN_CENTER',
    'LOCATION_TO_TOWNCENTER',
    'TAZ',
    'PARCEL_ACRES',
    'PARCEL_SQFT',
    'WITHIN_BONUSUNIT_BNDY',
    'WITHIN_TRPA_BNDY'
]

## Load Scenario Configuration

**Edit `config_path` below to point at your scenario config file. All other cells run automatically.**

In [ ]:
# ── EDIT THIS PATH ────────────────────────────────────────────────────────────
config_path = r"C:\path\to\your\scenario_config.json"
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# Load and parse config
with open(config_path, 'r') as f:
    config = json.load(f)

config_fname = Path(config_path).name

scenario_name           = config['scenario_name']
adjust_for_workfromhome = config.get('adjust_for_workfromhome', 'no').strip().lower() == 'yes'
adjust_occupancy        = config.get('adjust_occupancy', 'no').strip().lower() == 'yes'
adjust_taz_pop          = config.get('adjust_taz_population', 'no').strip().lower() == 'yes'
zone_proportions        = config['zone_proportions']
occupancy_rates         = config['occupancy_rates']
income_proportions      = config['income_proportions']

if adjust_occupancy:
    additional_occupied_units = int(config['adjust_occupancy_additional_units'])
if adjust_taz_pop:
    target_population_2035 = int(config['target_population_2035'])
    target_population_2050 = int(config['target_population_2050'])

print(f"Scenario:                 {scenario_name}")
print(f"Config loaded from:       {config_path}")
print(f"Work-from-home adjust:    {adjust_for_workfromhome}")
print(f"Occupancy adjust:         {adjust_occupancy}")
if adjust_occupancy:
    print(f"Additional occ units:     {additional_occupied_units}")
print(f"TAZ population adjust:    {adjust_taz_pop}")
if adjust_taz_pop:
    print(f"Target pop 2035:          {target_population_2035}")
    print(f"Target pop 2050:          {target_population_2050}")

## Establish Scenario Conditions

In [ ]:
folder_2035 = out_dir / (scenario_name + '2035')
folder_2050 = out_dir / (scenario_name + '2050')
# create output folders if they don't exist
folder_2035.mkdir(parents=True, exist_ok=True)
folder_2050.mkdir(parents=True, exist_ok=True)
print(f"Output 2035: {folder_2035}")
print(f"Output 2050: {folder_2050}")

### Get base year parcel data and scenario residential zone development targets

In [ ]:
# Load base parcel data
parcel_data_path = base_data_dir / 'base_parcel_data.parquet'
sdfParcels = pd.read_parquet(parcel_data_path)

### Establish scenario proportions

In [ ]:
# zone_proportions, occupancy_rates, and income_proportions are loaded from config above.
# Displayed here for reference:
print("Zone proportions:")
for k, v in zone_proportions.items():
    print(f"  {k}: {v}")
print("\nOccupancy rates:")
for k, v in occupancy_rates.items():
    print(f"  {k}: {v}")
print("\nIncome proportions:")
for k, v in income_proportions.items():
    print(f"  {k}: {v}")

# Build residential unit lookup and pool from config
dfResZoned = pd.DataFrame(config['residential_units'])
dfPool     = pd.DataFrame(config['residential_units'])
dfPool     = get_adjusted_future_units(dfPool, zone_proportions)

In [ ]:
conditions = get_parcel_conditions()
sdfParcels, df_built_parcels = forecast_jurisdiction_pools(sdfParcels, dfPool, conditions)
sdfParcels, df_built_parcels = forecast_trpa_pools(sdfParcels, dfPool, conditions, df_built_parcels)
sdfParcels, df_built_parcels = assign_remainders(sdfParcels, conditions, df_built_parcels, adu_target=4385)
df_res_assigned_lookup = data_dir / "inputs/res_assigned_lookup.csv"
df_forecast_check = check_forecast_results(sdfParcels, dfPool)
sdfParcels = assign_development_year(sdfParcels, dfResZoned)
sdfParcels, dfResAssigned = assign_occupancy_rate(sdfParcels, occupancy_rates, df_res_assigned_lookup)
sdfParcels = assign_income_categories(sdfParcels, dfResAssigned, income_proportions)

In [ ]:
# Base year SocioEcon
socio      = r'Base\data\processed_data\SocioEcon_Summer.csv'
socio_path = local_path.parents[1].joinpath(socio)
dfSocio    = pd.read_csv(socio_path)
dfSocio.rename(columns={'taz': 'TAZ'}, inplace=True)
dfSocio['TAZ'] = dfSocio['TAZ'].astype(str)

# Base year school enrollment
school      = r'Base\data\processed_data\SchoolEnrollment.csv'
school_path = local_path.parents[1].joinpath(school)
dfSchool    = pd.read_csv(school_path)

In [ ]:
dfTAZ_Summary_2035, dfTAZ_Summary_2050 = build_taz_summary(sdfParcels, dfSocio)

# Save intermediate parcel and TAZ outputs to a scenario-named subfolder
date_str    = datetime.now().strftime("%Y-%m-%d")
data_folder = Path('data') / scenario_name
data_folder.mkdir(parents=True, exist_ok=True)

sdfParcels.to_parquet(data_folder / f'sdfParcels_{date_str}.parquet', index=False)
dfTAZ_Summary_2035.to_csv(data_folder / f'TAZ_Summary_2035_{date_str}.csv', index=False)
dfTAZ_Summary_2050.to_csv(data_folder / f'TAZ_Summary_2050_{date_str}.csv', index=False)

In [ ]:
taz_field_mapping = {
    'TAZ':                              'TAZ',
    'TOTAL_FORECASTED_RESIDENTIAL_UNITS': 'total_residential_units',
    'NEW_OCCUPANCY_RATE':               'census_occ_rate',
    'TOTAL_FORECASTED_UNITS_OCCUPIED':  'total_occ_units',
    'TOTAL_FORECASTED_UNITS_LOW_INCOME': 'occ_units_low_inc',
    'TOTAL_FORECASTED_UNITS_MED_INCOME': 'occ_units_med_inc',
    'TOTAL_FORECASTED_UNITS_HIGH_INCOME': 'occ_units_high_inc',
    'persons_per_occ_unit':             'persons_per_occ_unit',
    'TOTAL_FORECASTED_PERSONS':         'total_persons',
    'emp_retail':                       'emp_retail',
    'emp_srvc':                         'emp_srvc',
    'emp_rec':                          'emp_rec',
    'emp_game':                         'emp_game',
    'emp_other':                        'emp_other'
}
df_taz_2035 = clean_taz_summary(dfTAZ_Summary_2035, taz_field_mapping)
df_taz_2050 = clean_taz_summary(dfTAZ_Summary_2050, taz_field_mapping)

In [ ]:
# Adjust persons per occupied unit — two modes:
#   adjust_taz_population=yes  → fit to regional population targets, then save the
#                                persons_per_occ_unit lookup for downstream scenarios
#   (default)                  → apply the existing persons_per_occ_unit lookup
persons_per_occ_unit_2035_path = local_path.parents[0] / 'data/processed_data/base_data/persons_per_occ_unit_2035.csv'
persons_per_occ_unit_2050_path = local_path.parents[0] / 'data/processed_data/base_data/persons_per_occ_unit_2050.csv'

if adjust_taz_pop:
    df_taz_2035 = adjust_taz_population(df_taz_2035, target_population=target_population_2035)
    df_taz_2050 = adjust_taz_population(df_taz_2050, target_population=target_population_2050)
    df_taz_2035[['TAZ', 'persons_per_occ_unit']].to_csv(persons_per_occ_unit_2035_path, index=False)
    df_taz_2050[['TAZ', 'persons_per_occ_unit']].to_csv(persons_per_occ_unit_2050_path, index=False)
    print(f"persons_per_occ_unit lookup saved to {persons_per_occ_unit_2035_path.parent}")
else:
    persons_per_occ_unit_2035 = pd.read_csv(persons_per_occ_unit_2035_path)
    persons_per_occ_unit_2050 = pd.read_csv(persons_per_occ_unit_2050_path)
    df_taz_2035 = adjust_taz_persons_per_occ_unit(df_taz_2035, persons_per_occ_unit_2035)
    df_taz_2050 = adjust_taz_persons_per_occ_unit(df_taz_2050, persons_per_occ_unit_2050)

## Optional: Adjust Occupancy to Target Units

Controlled by `adjust_occupancy` in config. If `"yes"`, proportionally scales occupied units by `adjust_occupancy_additional_units`, distributing the extra units equally between medium and high income.

In [ ]:
def adjust_occupied_units(df, additional_units, label, rng):
    """
    Proportionally scale total_occ_units upward by additional_units, distributing
    the extra units equally between medium and high income. Recomputes total_persons,
    rounds all counts to integers, then corrects rounding residuals so regional totals
    and per-TAZ income sums match exactly.
    """
    target_sum      = int(df["total_occ_units"].sum()) + additional_units
    additional_med  = additional_units / 2
    additional_high = additional_units / 2

    # Scale total occupied units
    occ_factor = target_sum / df["total_occ_units"].sum()
    df["total_occ_units"] = df["total_occ_units"] * occ_factor

    # Scale medium income
    current_med = df["occ_units_med_inc"].sum()
    df["occ_units_med_inc"] = df["occ_units_med_inc"] * (current_med + additional_med) / current_med

    # Scale high income
    current_high = df["occ_units_high_inc"].sum()
    df["occ_units_high_inc"] = df["occ_units_high_inc"] * (current_high + additional_high) / current_high

    # Recompute persons before rounding
    df["total_persons"] = (df["total_occ_units"] * df["persons_per_occ_unit"]).round().astype(int)

    # Round all occ unit columns to int
    for col in ["total_occ_units", "occ_units_med_inc", "occ_units_high_inc"]:
        df[col] = df[col].round().astype(int)

    # Fix rounding residual in total_occ_units
    diff = target_sum - df["total_occ_units"].sum()
    if diff != 0:
        direction = 1 if diff > 0 else -1
        eligible  = df.index if direction == 1 else df.index[df["total_occ_units"] > 0]
        chosen    = rng.choice(eligible, size=abs(diff), replace=False)
        df.loc[chosen, "total_occ_units"] += direction
        print(f"  {label} total_occ_units adjusted by {diff:+d} units across {abs(diff)} random TAZs")

    # Fix per-TAZ income class mismatch (absorb into high income)
    income_sum    = df["occ_units_low_inc"] + df["occ_units_med_inc"] + df["occ_units_high_inc"]
    taz_diff      = df["total_occ_units"] - income_sum
    mismatch_count = (taz_diff != 0).sum()
    if mismatch_count > 0:
        df["occ_units_high_inc"] = df["occ_units_high_inc"] + taz_diff
        print(f"  {label}: income mismatch corrected in {mismatch_count} TAZs (adjusted occ_units_high_inc)")

    # Verify
    income_sum = df["occ_units_low_inc"] + df["occ_units_med_inc"] + df["occ_units_high_inc"]
    bad = (income_sum != df["total_occ_units"]).sum()
    print(f"{label} — occupied: {df['total_occ_units'].sum()} (target: {target_sum})  |  "
          f"income sum check: {'PASS' if bad == 0 else f'FAIL — {bad} TAZs mismatched'}")
    return df


if adjust_occupancy:
    rng = np.random.default_rng(seed=42)
    df_taz_2035 = adjust_occupied_units(df_taz_2035, additional_occupied_units, "2035", rng)
    df_taz_2050 = adjust_occupied_units(df_taz_2050, additional_occupied_units, "2050", rng)
else:
    print("Occupancy adjustment skipped (adjust_occupancy = no in config)")

In [ ]:
dfSchool_2035, dfSchool_2050 = adjust_school_enrollment(df_taz_2035, df_taz_2050, dfSocio, dfSchool)

## Optional: Work From Home Employment Adjustment

Controlled by `adjust_for_workfromhome` in config.

In [ ]:
if adjust_for_workfromhome:
    employment_2035_path = local_path.parents[0] / r'data/inputs/employment_2035.csv'
    employment_2050_path = local_path.parents[0] / r'data/inputs/employment_2050.csv'
    dfEmployment_2035 = pd.read_csv(employment_2035_path)
    dfEmployment_2050 = pd.read_csv(employment_2050_path)
    df_taz_2035 = update_employment_numbers(df_taz_2035, dfEmployment_2035)
    df_taz_2050 = update_employment_numbers(df_taz_2050, dfEmployment_2050)
else:
    print("Work-from-home adjustment skipped (adjust_for_workfromhome = no in config)")

In [ ]:
# rename TAZ to taz for model input
df_taz_2035.rename(columns={'TAZ': 'taz'}, inplace=True)
df_taz_2050.rename(columns={'TAZ': 'taz'}, inplace=True)

In [ ]:
# Static inputs that don't change between scenarios
overnight_visitors_path = local_path.parents[0] / r'data/inputs/OvernightVisitorZonalData_Summer.csv'
dfOvernightVisitors     = pd.read_csv(overnight_visitors_path)

visitor_occupancy_path  = local_path.parents[1] / r'Base/data/processed_data/VisitorOccupancyRates_Summer.csv'
dfVisitorOccupancy      = pd.read_csv(visitor_occupancy_path)

# Export TAZ summaries
df_taz_2035.to_csv(folder_2035 / 'SocioEcon_Summer.csv',              index=False)
dfSchool_2035.to_csv(folder_2035 / 'SchoolEnrollment.csv',            index=False)
dfOvernightVisitors.to_csv(folder_2035 / 'OvernightVisitorZonalData_Summer.csv', index=False)
dfVisitorOccupancy.to_csv(folder_2035 / 'VisitorOccupancyRates_Summer.csv',      index=False)

df_taz_2050.to_csv(folder_2050 / 'SocioEcon_Summer.csv',              index=False)
dfSchool_2050.to_csv(folder_2050 / 'SchoolEnrollment.csv',            index=False)
dfOvernightVisitors.to_csv(folder_2050 / 'OvernightVisitorZonalData_Summer.csv', index=False)
dfVisitorOccupancy.to_csv(folder_2050 / 'VisitorOccupancyRates_Summer.csv',      index=False)

# Copy config file to both output folders for reproducibility
shutil.copy(config_path, folder_2035 / config_fname)
shutil.copy(config_path, folder_2050 / config_fname)

print(f"\nScenario '{scenario_name}' complete.")
print(f"  2035 outputs: {folder_2035}")
print(f"  2050 outputs: {folder_2050}")
print(f"  Config copied to both output folders as: {config_fname}")